## Permutation Test 

#### Import libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline 
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer
import shap
from imblearn.ensemble import BalancedRandomForestClassifier
from matplotlib.patches import Patch
from sklearn.inspection import PartialDependenceDisplay
from sklearn.tree import plot_tree
from itertools import combinations
from sklearn.tree import plot_tree
from sklearn.calibration import CalibratedClassifierCV

#### Load Data and Preprocessing

In [2]:
file_path = "https://raw.githubusercontent.com/sarahvastani/portfolio/refs/heads/main/Survival_of_Migration_Project/Dataset/Dataset1.phiSurvival.20250707.csv"
df = pd.read_csv(file_path)
df

,reference,name_in_vcf,release_site,capture_site,release_year,tag_type,sex_binary,age_release,release_gps.n,release_gps.w,...,range_norm_p9,range_norm_p10,range_norm_releaseDay,range_norm_fall_detectDay1,range_norm_fall_bearing1,range_norm_bodyCondition,ch_10days,ch,phi,phi_binary
0,AH17K18,AH17K18,Pemberton,Pemberton,2019,radio,1,HY,50.353386,-122.832926,...,0.666667,NaN,0.000000,NaN,NaN,0.849981,days_100110000000000000000000000000,100110000000000000000000000000,0.000000e+00,0
1,AH18K02,AH18K02,Pemberton,Pemberton,2019,radio,1,HY,50.353386,-122.832926,...,0.833333,NaN,0.043478,0.250,0.424781,0.674626,days_110000100000000000000000000000,110000100000000000000000000000,1.084520e-38,0
2,AH22K02,AH22K02,Pemberton,Pemberton,2019,radio,1,HY,50.353386,-122.832926,...,0.388889,NaN,0.217391,0.500,0.731704,0.347152,days_101000000000000000000000000000,101000000000000000000000000000,8.884759e-14,0
3,AH22K04,AH22K04_S154_L001,Pemberton,Pemberton,2019,radio,1,HY,50.353386,-122.832926,...,0.444444,NaN,0.217391,NaN,NaN,0.571480,days_100000000000000000000000000000,100000000000000000000000000000,1.285971e-240,0
4,AH22K09,AH22K09,Pemberton,Pemberton,2019,radio,1,HY,50.353386,-122.832926,...,0.444444,NaN,0.217391,0.550,0.836626,0.294291,days_101000000000000000000000000000,101000000000000000000000000000,8.884759e-14,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,EH31H02,EH31H02,Pemberton,Pemberton,2023,radio,1,HY,50.220153,-122.884084,...,0.777778,0.375,0.608696,0.600,0.474724,0.468136,days_101001000000000000000000000000,101001000000000000000000000000,0.000000e+00,0
475,EH31H03,EH31H03,Pemberton,Pemberton,2023,radio,1,HY,50.220153,-122.884084,...,0.500000,0.250,0.608696,0.825,0.474724,0.468227,days_101100000000000000000000000000,101100000000000000000000000000,2.362661e-23,0
476,EH31H07,EH31H07,Pemberton,Pemberton,2023,radio,0,HY,50.220153,-122.884084,...,0.555556,0.500,0.608696,0.900,0.474724,0.281292,days_100110000000000000000000000000,100110000000000000000000000000,0.000000e+00,0
477,EH31H08,EH31H08,Pemberton,Pemberton,2023,radio,0,HY,50.220153,-122.884084,...,0.444444,0.250,0.608696,0.425,0.352253,0.513245,days_110100100000000000000000100111,110100100000000000000000100111,1.000000e+00,1


#### Model Training

In [3]:
# Feature and label setup
features = ['fall_detectDay1', 'fall_bearing1', 'distal', 'p10', 'bodyCondition',
            'tarsus.length', 'kipps', 'tail.length', 'wing.cord', 
            'sex_binary', 'releaseDay', 'aims_heterozygosity', 
            'aims_ancestry', 'release_year']

X = df[features]
y = df['phi_binary'].astype(int)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=120, 
    random_state=42,
    stratify=y
)


# Pipeline setup
pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('smote', SMOTE(random_state=42)),
    ('brf', BalancedRandomForestClassifier(random_state=42))
])

# Parameter grid
param_grid = {
    'brf__n_estimators': [100, 200, 300],
    'brf__max_depth': [10, 20, None],
    'brf__max_features': ['sqrt', 'log2', None],
    'brf__min_samples_split': [2, 5, 10],
    'brf__min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=40,
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the model
search.fit(X_train, y_train)
best_model = search.best_estimator_
probs = best_model.predict_proba(X_test)[:, 1]

# Threshold tuning
thresholds = np.linspace(0.4, 0.7, 100)
results = []

for t in thresholds:
    preds = (probs >= t).astype(int)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    results.append({
        'threshold': t,
        'accuracy': report['accuracy'],
        'class0_recall': report['0']['recall'],
        'class1_recall': report['1']['recall'],
        'class0_f1': report['0']['f1-score'],
        'class1_f1': report['1']['f1-score'],
        'macro_f1': (report['0']['f1-score'] + report['1']['f1-score']) / 2
    })

# Optimal threshold
results_df = pd.DataFrame(results)
best_row = results_df.sort_values(by='macro_f1', ascending=False).iloc[0]
best_thresh = best_row['threshold']
final_preds = (probs >= best_thresh).astype(int)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `False` to silence this warning and adopt the future behaviour.


#### Evaluation

In [8]:
print(confusion_matrix(y_test, final_preds))
print(classification_report(y_test, final_preds, zero_division=0))

[[68 15]
 [19 18]]
              precision    recall  f1-score   support

           0       0.78      0.82      0.80        83
           1       0.55      0.49      0.51        37

    accuracy                           0.72       120
   macro avg       0.66      0.65      0.66       120
weighted avg       0.71      0.72      0.71       120



In [11]:
# Settings

N_PERMUTATIONS = 200   # 100–200 is standard
np.random.seed(1)

# Baseline Model + Data

rf_model = best_model.named_steps['brf']

X_test_imputed = best_model.named_steps['imputer'].transform(X_test)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=features)

baseline_probs = rf_model.predict_proba(X_test_imputed.values)[:, 1]
baseline_preds = (baseline_probs >= best_thresh).astype(int)
baseline_score = f1_score(y_test, baseline_preds, average='macro')

print(f"Baseline macro F1: {baseline_score:.4f}")

# Repeated Pairwise Permutation Importance

pair_results = []
feature_pairs = list(combinations(features, 2))

for feat1, feat2 in feature_pairs:
    drops = []

    for _ in range(N_PERMUTATIONS):
        X_perm = X_test_imputed.copy()

        # Jointly permute both features
        perm = np.random.permutation(len(X_perm))
        X_perm[[feat1, feat2]] = X_perm[[feat1, feat2]].iloc[perm].values

        # Compute F1 after permutation
        perm_probs = rf_model.predict_proba(X_perm.values)[:, 1]
        perm_preds = (perm_probs >= best_thresh).astype(int)
        perm_score = f1_score(y_test, perm_preds, average='macro')

        drops.append(baseline_score - perm_score)

    # Summary statistics
    mean_drop = np.mean(drops)
    q2_5 = np.quantile(drops, 0.025)
    q97_5 = np.quantile(drops, 0.975)

    pair_results.append({
        "Feature1": feat1,
        "Feature2": feat2,
        "Mean_Delta_F1": mean_drop,
        "Q2.5": q2_5,
        "Q97.5": q97_5
    })

# Save Results

pair_df = pd.DataFrame(pair_results)
pair_df = pair_df.sort_values("Mean_Delta_F1", ascending=False)

pair_df.to_excel("pairwise_permutation_with_quantiles.xlsx", index=False)

print("Saved results to pairwise_permutation_with_quantiles.xlsx")
pair_df

Baseline macro F1: 0.6571
Saved results to pairwise_permutation_with_quantiles.xlsx


,Feature1,Feature2,Mean_Delta_F1,Q2.5,Q97.5
24,fall_bearing1,release_year,0.163635,0.089572,0.235038
89,aims_heterozygosity,release_year,0.153119,0.082711,0.220847
90,aims_ancestry,release_year,0.141158,0.063763,0.211474
54,bodyCondition,release_year,0.136227,0.060877,0.222639
62,tarsus.length,release_year,0.136010,0.066389,0.211128
...,...,...,...,...,...
70,tail.length,wing.cord,0.049329,0.000000,0.096014
29,distal,tail.length,0.046505,0.014256,0.080696
6,fall_detectDay1,tail.length,0.045309,-0.008479,0.102316
81,sex_binary,releaseDay,0.043636,0.007034,0.093671


In [10]:
# Settings

N_PERMUTATIONS = 1000   
np.random.seed(1)

# Baseline Model + Data

rf_model = best_model.named_steps['brf']

X_test_imputed = best_model.named_steps['imputer'].transform(X_test)
X_test_imputed = pd.DataFrame(X_test_imputed, columns=features)

baseline_probs = rf_model.predict_proba(X_test_imputed.values)[:, 1]
baseline_preds = (baseline_probs >= best_thresh).astype(int)
baseline_score = f1_score(y_test, baseline_preds, average='macro')

print(f"Baseline macro F1: {baseline_score:.4f}")

# Repeated Permutation Importance

results = []

for feat in features:
    drops = []

    for _ in range(N_PERMUTATIONS):
        X_perm = X_test_imputed.copy()

        # Permute this feature
        perm = np.random.permutation(len(X_perm))
        X_perm[feat] = X_perm[feat].iloc[perm].values

        # Compute F1 after permutation
        perm_probs = rf_model.predict_proba(X_perm.values)[:, 1]
        perm_preds = (perm_probs >= best_thresh).astype(int)
        perm_score = f1_score(y_test, perm_preds, average='macro')

        drops.append(baseline_score - perm_score)

    # Summary statistics
    mean_drop = np.mean(drops)
    q2_5 = np.quantile(drops, 0.025)
    q97_5 = np.quantile(drops, 0.975)

    results.append({
        "Feature": feat,
        "Mean_Delta_F1": mean_drop,
        "Q2.5": q2_5,
        "Q97.5": q97_5
    })

# -----------------------------------
# SAVE RESULTS
# -----------------------------------
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Mean_Delta_F1", ascending=False)

results_df.to_excel("univariate_permutation_with_quantiles.xlsx", index=False)

print("Saved results to univariate_permutation_with_quantiles.xlsx")
results_df

Baseline macro F1: 0.6571
Saved results to univariate_permutation_with_quantiles.xlsx


,Feature,Mean_Delta_F1,Q2.5,Q97.5
13,release_year,0.109199,0.034921,0.192857
1,fall_bearing1,0.088652,0.021346,0.149619
4,bodyCondition,0.075525,0.014256,0.133333
11,aims_heterozygosity,0.070336,0.019359,0.123810
5,tarsus.length,0.061144,0.026500,0.096014
6,kipps,0.060339,0.026500,0.096014
2,distal,0.059666,0.026500,0.094468
12,aims_ancestry,0.057338,0.012420,0.104586
8,wing.cord,0.054148,0.012268,0.098359
3,p10,0.053709,0.019493,0.094468
